# tl-jockey — Train Grounding Head on Charades-STA (Colab)

Trains the multimodal grounding head (`GroundingHead`) on Charades-STA precomputed features.

**Prerequisite**: Charades videos downloaded + features extracted to `Drive/tl_jockey/features/charades/<vid>.npz` and query cache at `Drive/tl_jockey/features/charades/query_emb.npz`.

Runtime: GPU (T4 / P100 / A100) recommended. With frozen features the bottleneck is the small fusion transformer, not the encoders.

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
WORK_DIR = '/content/tl-jockey'
DRIVE_ROOT = '/content/drive/MyDrive/tl_jockey'

if not os.path.isdir(WORK_DIR):
    # Replace with your fork URL
    subprocess.run(['git', 'clone', 'https://github.com/<your-user>/tl-jockey', WORK_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=False)

%cd $WORK_DIR
!ls

## 2. Install dependencies

In [ ]:
!pip install -q torch numpy decord scenedetect[opencv] qdrant-client openai huggingface_hub transformers

## 3. Set API keys (only needed if extracting features here, not for training on cached features)

In [ ]:
from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.environ['HF_API_KEY']         = userdata.get('HF_API_KEY')
os.environ['HF_TOKEN']           = os.environ['HF_API_KEY']

## 4. (Optional) Download Charades-STA annotations

If not already on Drive.

In [ ]:
ANN_DIR = f'{DRIVE_ROOT}/data'
os.makedirs(ANN_DIR, exist_ok=True)

from jockey.open_source.training.charades_sta import download_annotations
paths = download_annotations(ANN_DIR)
print(paths)

## 5. (One-time) Precompute query embeddings

Embeds all unique queries via OpenAI `text-embedding-3-large` (OpenRouter). Cache to Drive — only runs once.

In [ ]:
FEATURES_DIR = f'{DRIVE_ROOT}/features/charades'
QUERY_CACHE  = f'{FEATURES_DIR}/query_emb.npz'
os.makedirs(FEATURES_DIR, exist_ok=True)

!python -m jockey.open_source.training.precompute_queries \
    --annotations {ANN_DIR}/charades_sta_train.txt {ANN_DIR}/charades_sta_test.txt \
    --out {QUERY_CACHE}

## 6. Train the grounding head

Adjust hyperparameters per ablation. Examples:
- Default (all modalities): no extra flags
- Vision-only ablation:     `--no-audio --no-caption --no-global`
- Deeper fusion:             `--num-layers 6`

In [ ]:
RUN = 'exp1_full'
OUT_DIR = f'{DRIVE_ROOT}/runs/{RUN}'

!python -m jockey.open_source.training.train \
    --features-dir {FEATURES_DIR} \
    --train-ann   {ANN_DIR}/charades_sta_train.txt \
    --test-ann    {ANN_DIR}/charades_sta_test.txt \
    --query-cache {QUERY_CACHE} \
    --out-dir     {OUT_DIR} \
    --hidden-dim 512 --num-layers 4 --num-heads 8 \
    --batch-size 16 --epochs 30 --lr 1e-4 \
    --mixed-precision \
    --num-workers 2 --device cuda \
    --log-every 50

## 7. Plot training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

tr = pd.read_csv(f'{OUT_DIR}/train_log.csv')
va = pd.read_csv(f'{OUT_DIR}/val_log.csv')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tr['step'], tr['loss'], label='total', alpha=0.5)
ax[0].plot(tr['step'], tr['rel'],  label='rel BCE')
ax[0].plot(tr['step'], tr['l1'],   label='boundary L1')
ax[0].plot(tr['step'], tr['iou_loss'], label='1-IoU')
ax[0].legend(); ax[0].set_xlabel('step'); ax[0].set_title('train losses')
ax[1].plot(va['epoch'], va['r03'], label='R@1@IoU=0.3')
ax[1].plot(va['epoch'], va['r05'], label='R@1@IoU=0.5')
ax[1].plot(va['epoch'], va['r07'], label='R@1@IoU=0.7')
ax[1].plot(va['epoch'], va['mIoU'], label='mIoU', linestyle='--')
ax[1].legend(); ax[1].set_xlabel('epoch'); ax[1].set_title('val metrics')
plt.tight_layout(); plt.show()

print(va.tail())

## 8. Resume from checkpoint

Colab sessions time out. To continue from `last.pt`:

In [ ]:
!python -m jockey.open_source.training.train \
    --features-dir {FEATURES_DIR} \
    --train-ann   {ANN_DIR}/charades_sta_train.txt \
    --test-ann    {ANN_DIR}/charades_sta_test.txt \
    --query-cache {QUERY_CACHE} \
    --out-dir     {OUT_DIR} \
    --resume      {OUT_DIR}/last.pt \
    --epochs 30 --batch-size 16 --lr 1e-4 \
    --mixed-precision --device cuda